In [13]:
import torch
from torch import nn
from transformers import RobertaTokenizer, RobertaForSequenceClassification
import torch.nn.functional as F  # For softmax
import matplotlib.pyplot as plt

# Step 1: Define the CodeBERT model with Dropout
class CodeBERTClassifier(nn.Module):
    def __init__(self):
        super(CodeBERTClassifier, self).__init__()
        self.model = RobertaForSequenceClassification.from_pretrained("microsoft/codebert-base", num_labels=2)
        self.dropout = nn.Dropout(p=0.3)  # Add a dropout layer

    def forward(self, input_ids, attention_mask=None):
        outputs = self.model(input_ids, attention_mask=attention_mask)
        logits = self.dropout(outputs.logits)  # Apply dropout
        return logits

# Step 2: Load the trained model
model = CodeBERTClassifier()
# model.load_state_dict(torch.load('E:/python-projects/llm/Trained_models/codebert_model.pth'))
model.load_state_dict(torch.load('C:/Users/ABHIMANYU/Downloads/codebert_classifier.pth'))

model.eval()  # Set the model to evaluation mode

# Move model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params}")

# Step 3: Load the tokenizer
tokenizer = RobertaTokenizer.from_pretrained("microsoft/codebert-base")

# Step 4: Preprocess the input code samples
def preprocess_input_code(code_samples):
    tokenized_samples = []
    attention_masks = []

    for code_sample in code_samples:
        tokenized_input = tokenizer(
            code_sample,
            padding='max_length',
            truncation=True,
            max_length=512,
            return_tensors='pt'
        )
        tokenized_samples.append(tokenized_input['input_ids'].squeeze(0))
        attention_masks.append(tokenized_input['attention_mask'].squeeze(0))

    # Convert to PyTorch tensors
    tokens = torch.stack(tokenized_samples)
    masks = torch.stack(attention_masks)

    return tokens, masks

# Step 5: Make predictions with probabilities
def predict_code_samples(model, code_samples):
    tokens, masks = preprocess_input_code(code_samples)
    
    # Move input tensors to the same device as the model
    tokens = tokens.to(device)
    masks = masks.to(device)

    with torch.no_grad():
        outputs = model(tokens, attention_mask=masks)
        # Apply softmax to get probabilities
        probabilities = torch.nn.functional.softmax(outputs, dim=1)

    return probabilities.cpu().numpy()  # Return probabilities as numpy array

# Step 6: Test the model with new samples
# new_code_samples ="""
# from typing import List

# class Solution:
#     def twoSum(self, nums: List[int], target: int) -> List[int]:
#         # Dictionary to store the value and its index
#         num_to_index = {}
        
#         # Iterate through the array
#         for i, num in enumerate(nums):
#             complement = target - num
            
#             # Check if the complement is already in the dictionary
#             if complement in num_to_index:
#                 return [num_to_index[complement], i]
            
#             # Store the current number and its index in the dictionary
#             num_to_index[num] = i
# """
# Load the text file with multiple code snippets
with open("sample_code.txt", "r") as file:
    file_content = file.read()

# Split based on delimiter '###'
code_samples = file_content.split("####")
code_samples = [code.strip() for code in code_samples if code.strip()]  # Clean up

# Make predictions
probabilities = predict_code_samples(model, code_samples)

# Print results with percentages for each code sample
for idx, (code, prob) in enumerate(zip(code_samples, probabilities)):
    ai_generated_prob = prob[1] * 100  # Percentage for AI-generated class (label 1)
    human_generated_prob = prob[0] * 100  # Percentage for Human-written class (label 0)
    print(f"Sample {idx+1}:")
    print(f"Prediction: {ai_generated_prob:.2f}% AI-generated, {human_generated_prob:.2f}% Human-generated\n")



Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at microsoft/codebert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Total parameters: 124647170
Sample 1:
Prediction: 2.02% AI-generated, 97.98% Human-generated

Sample 2:
Prediction: 9.85% AI-generated, 90.15% Human-generated

Sample 3:
Prediction: 98.05% AI-generated, 1.95% Human-generated

Sample 4:
Prediction: 99.76% AI-generated, 0.24% Human-generated

Sample 5:
Prediction: 1.35% AI-generated, 98.65% Human-generated

Sample 6:
Prediction: 99.89% AI-generated, 0.11% Human-generated

Sample 7:
Prediction: 99.83% AI-generated, 0.17% Human-generated

